# NB-08_salida_cnf_trim_dip

Process Flow SAS: **Salida_CNF_Trim_DIP** — `ProcessFlowContainer-hH8XWcN3BDAlowaA`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-08_salida_cnf_trim_dip.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## OPC TC Mixtas_M_DIP

Redistribuye la ejecución DIPRES del año de transacciones de tipo mixto (M) desde la muestra hacia el resto de analíticos según ponderadores de estructura, y consolida el devengado total por trimestre/mes y clasificación excluyendo códigos SCN de exclusión

*confianza: medium · verificador: approve · SAS: PROC SQL: múltiples LEFT JOIN + agregaciones (SUM/GROUP BY) encadenadas, redistribución de muestra a expansión y salida agregada*

In [ ]:
# ========= OPC TC Mixtas_M_DIP =========
# DIPRES: CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO, MES y PACAPR faltante
# Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
# DROP de columnas de presupuesto: se recrea #ejec_dip sin PPTO_INICIAL/PPTO_ACTUAL/EJEC_ACUM
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_2"))
work_conn.execute(text("SELECT * INTO #ejec_dip_2 FROM #ejec_dip"))
work_conn.execute(text("ALTER TABLE #ejec_dip_2 DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, EJEC_ACUM"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
work_conn.execute(text("EXEC sp_rename '#ejec_dip_2', '#ejec_dip'"))
ejec_dip = pd.read_sql(text("SELECT * FROM #ejec_dip"), work_conn)
_log("ejec_dip", ejec_dip)


In [ ]:
# TCONS MIXTAS DE LA MUESTRA DIPRES
# En SAS un LEFT JOIN + WHERE sobre columnas del lado derecho (t2, el LEFT)
# equivale a un INNER JOIN: las no-match quedan NULL y el WHERE las descarta igual
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_prev_dip"))
sql_ejec_muestra_prev = """
SELECT t2.*,
    t1.C_SCN,
    t1.C_CAGENTE,
    CAST(t1.C_CUP AS char(4)) AS C_CUP,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m_prev_dip
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_dip t2
    ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
    AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','08','09') AND t1.ENTIDAD NOT IN (5601, 10201)
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,
    t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t2.C_ENTIDAD, t2.ACTIVIDAD, t2.NIVEL, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_muestra_prev))


In [ ]:
# DATA EJEC_MUESTRA_MIXTAS_M_DIP (DROP=DEVENG); SET EJEC_MUESTRA_MIXTAS_M_prev_DIP
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_dip"))
work_conn.execute(text("SELECT * INTO #ejec_muestra_mixtas_m_dip FROM #ejec_muestra_mixtas_m_prev_dip"))
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m_dip DROP COLUMN DEVENG"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_prev_dip"))
ejec_muestra_mixtas_m_dip = pd.read_sql(text("SELECT * FROM #ejec_muestra_mixtas_m_dip"), work_conn)
_log("ejec_muestra_mixtas_m_dip", ejec_muestra_mixtas_m_dip)


In [ ]:
# DIPRES: CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
# Faltante numérico de SAS (C_ENTIDAD=.) es NULL en SQL Server: se agrega explícito junto al NOT IN
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_dip"))
sql_cruce_ejec_exp = """
SELECT t2.*,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.ENTIDAD,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip t2
    ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
    AND (t2.C_ENTIDAD IS NOT NULL AND t2.C_ENTIDAD NOT IN (5601, 10201))
    AND t1.Clasif_Tcons = 'M'
"""
work_conn.execute(text(sql_cruce_ejec_exp))
cruce_ejec_exp_mixtas_m_dip = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mixtas_m_dip"), work_conn)
_log("cruce_ejec_exp_mixtas_m_dip", cruce_ejec_exp_mixtas_m_dip)


In [ ]:
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), 5216
# PARA EJECUCION AÑO t
# WHERE t2.DEVENG<>. -> faltante de SAS es NULL en SQL Server: se traduce como IS NOT NULL
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
sql_ejec_exp_prelim = """
SELECT t2.*,
    t1.C_SCN,
    t1.C_CUP,
    t1.C_CAGENTE,
    SUM(t2.DEVENG) AS DATO_OR,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim_dip
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m_dip t2
    ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.MES = t1.MES AND t2.ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,
    t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t2.C_ENTIDAD, t2.ACTIVIDAD, t2.NIVEL, t2.ENTIDAD, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_exp_prelim))


In [ ]:
# DATA EJEC_EXP_MIXTAS_M_DIP (DROP=DEVENG DATO_OR Clasif_Tcons ENTIDAD); SET EJEC_EXP_MIXTAS_M_prelim_DIP
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_dip"))
work_conn.execute(text("SELECT * INTO #ejec_exp_mixtas_m_dip FROM #ejec_exp_mixtas_m_prelim_dip"))
work_conn.execute(text("ALTER TABLE #ejec_exp_mixtas_m_dip DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons, ENTIDAD"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
ejec_exp_mixtas_m_dip = pd.read_sql(text("SELECT * FROM #ejec_exp_mixtas_m_dip"), work_conn)
_log("ejec_exp_mixtas_m_dip", ejec_exp_mixtas_m_dip)


In [ ]:
# Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto
# SET concatena ambas fuentes; UNION ALL en SQL Server exige mismas columnas -> se
# alinean por la lista de columnas de la primera fuente
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total_dip_2"))
cols_muestra = list(ejec_muestra_mixtas_m_dip.columns)
cols_comunes = ", ".join(cols_muestra)
sql_total_2 = f"""
SELECT {cols_comunes} INTO #ejec_mixtas_m_total_dip_2 FROM #ejec_muestra_mixtas_m_dip
UNION ALL
SELECT {cols_comunes} FROM #ejec_exp_mixtas_m_dip
"""
work_conn.execute(text(sql_total_2))


In [ ]:
# DIPRES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF trim
# TRIM: función pura del GROUP BY (MES), se calcula sobre el resultado agregado
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mixt_aper_dip"))
sql_aper = """
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_CUP,
    SUM(t1.DEVENGO) AS Dato,
    t1.ACTIVIDAD,
    t1.NIVEL
INTO #ejec_y_mixt_aper_dip
FROM #ejec_mixtas_m_total_dip_2 t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_aper))
ejec_y_mixt_aper_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mixt_aper_dip"), work_conn)
_log("ejec_y_mixt_aper_dip", ejec_y_mixt_aper_dip)


In [ ]:
# salida agregada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mixt_dip"))
sql_agregado = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
    SUM(t1.Dato) AS Dato, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_y_mixt_dip
FROM #ejec_y_mixt_aper_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_CUP
"""
work_conn.execute(text(sql_agregado))
ejec_y_mixt_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mixt_dip"), work_conn)
_log("ejec_y_mixt_dip", ejec_y_mixt_dip)


In [ ]:
# DROP TABLE de limpieza final del SAS: en la sesión los DROP TABLE IF EXISTS previos
# ya cumplen ese rol; se liberan igualmente las #tmp intermedias que ya no se usan
for t in ["#ejec_muestra_mixtas_m_dip", "#cruce_ejec_exp_mixtas_m_dip", "#ejec_exp_mixtas_m_prelim_dip", "#ejec_exp_mixtas_m_dip", "#ejec_mixtas_m_total_dip_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## OPC TC MGCG_DIP

Redistribuye la ejecución de transacciones MGCG (dentro y fuera de muestra) según la estructura de ponderadores y prom_mes, y consolida la ejecución trimestral excluyendo códigos SCN financieros para el reporte CNF

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOIN con estructura de muestra + agregación, UNION ALL de muestra/expansión, y salida agregada por trimestre para CNF*

In [ ]:
# ========= OPC TC MGCG_DIP =========
# DIPRES: CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGCG SEGUN ESTRUCTURA, PARA EJECUCION AÑO, MES y PACAPR faltante
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgcg_prev_dip"))
sql_ejec_muestra_mgcg_prev_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG/1000000*t1.POND) AS DEVENGO
INTO #ejec_muestra_mgcg_prev_dip
FROM GOBGENER.dbo.EST_MGCG t1
LEFT JOIN #ejec_dip t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
        (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
WHERE t2.C_CUENTA IN ('24') AND t2.C_ITEM IN ('03') AND t1.ENTIDAD NOT IN (5601,10201)
GROUP BY t2.SUBSECTOR,t2.MONEDA,t2.MES,t2.AÑO,t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.C_ITEM,
         t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t2.C_ENTIDAD,t2.ACTIVIDAD,t2.NIVEL,t1.C_SCN,t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muestra_mgcg_prev_dip))


In [ ]:
# DATA EJEC_MUESTRA_MGCG_DIP (DROP=DEVENG); SET EJEC_MUESTRA_MGCG_prev_DIP
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgcg_dip"))
cols_ejec_muestra_mgcg_dip = """SUBSECTOR,MONEDA,MES,AÑO,C_PARTIDA,C_CAPITULO,C_PROGRAMA,C_TIPO_CUENTA,C_CUENTA,C_ITEM,
         C_ASIGNACION,C_ANALITICO,N_ANALITICO,C_ENTIDAD,ACTIVIDAD,NIVEL,C_SCN,C_CAGENTE,DEVENGO"""
sql_ejec_muestra_mgcg_dip = f"""
SELECT {cols_ejec_muestra_mgcg_dip}
INTO #ejec_muestra_mgcg_dip
FROM #ejec_muestra_mgcg_prev_dip
"""
work_conn.execute(text(sql_ejec_muestra_mgcg_dip))
# PROC SQL; DROP TABLE EJEC_MUESTRA_MGCG_prev_DIP
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgcg_prev_dip"))
ejec_muestra_mgcg_dip = pd.read_sql(text("SELECT * FROM #ejec_muestra_mgcg_dip"), work_conn)
_log("ejec_muestra_mgcg_dip", ejec_muestra_mgcg_dip)


In [ ]:
# DIPRES: CREA TABLA DE ANALITICOS DE MGCG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
# El faltante de SAS (t2.C_ENTIDAD=.) es NULL tras el LEFT JOIN en SQL Server
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgcg_dip"))
sql_cruce_ejec_exp_mgcg_dip = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgcg_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip t2 ON (t2.C_ENTIDAD=t1.ENTIDAD AND t2.C_ASIGNACION=t1.C_ASIGNACION AND t2.C_ANALITICO=t1.C_ANALITICO AND t2.C_CUENTA=t1.C_CUENTA AND
                            t2.C_ITEM=t1.C_ITEM)
WHERE t1.M_MUESTRA=0 AND t1.Clasif_Tcons='MGCG' AND (t2.C_ENTIDAD IS NULL OR t2.C_ENTIDAD NOT IN (5601,10201))
"""
work_conn.execute(text(sql_cruce_ejec_exp_mgcg_dip))
cruce_ejec_exp_mgcg_dip = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mgcg_dip"), work_conn)
_log("cruce_ejec_exp_mgcg_dip", cruce_ejec_exp_mgcg_dip)


In [ ]:
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), 5216
# PARA EJECUCION AÑO t
# PROM_MES ya está en la sesión como #prom_mes (WORK.PROM_MES de un nodo anterior);
# 'AÑO'n de SAS (nombre con caracter especial entre comillas) es simplemente la columna AÑO
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgcg_prelim_dip"))
sql_ejec_exp_mgcg_prelim_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mgcg_prelim_dip
FROM GOBGENER.dbo.EST_EXP_MGCG t1
LEFT JOIN #cruce_ejec_exp_mgcg_dip t2 ON (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
LEFT JOIN #prom_mes t3 ON (t2.MONEDA = t3.MONEDA) AND (t2.MES = t3.MES) AND (t2.AÑO = t3.AÑO)
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR,t2.MONEDA, t2.AÑO, t2.MES,t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.C_ITEM,
         t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t2.C_ENTIDAD,t2.ACTIVIDAD,t2.NIVEL,t2.ENTIDAD,t2.Clasif_Tcons,t1.C_SCN,
         t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_exp_mgcg_prelim_dip))


In [ ]:
# DATA EJEC_EXP_MGCG_DIP (DROP=DEVENG DATO_OR Clasif_Tcons ENTIDAD); SET EJEC_EXP_MGCG_prelim_DIP
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgcg_dip"))
cols_ejec_exp_mgcg_dip = """SUBSECTOR,MONEDA,MES,AÑO,C_PARTIDA,C_CAPITULO,C_PROGRAMA,C_TIPO_CUENTA,C_CUENTA,C_ITEM,
         C_ASIGNACION,C_ANALITICO,N_ANALITICO,C_ENTIDAD,ACTIVIDAD,NIVEL,C_SCN,C_CAGENTE,DEVENGO"""
sql_ejec_exp_mgcg_dip = f"""
SELECT {cols_ejec_exp_mgcg_dip}
INTO #ejec_exp_mgcg_dip
FROM #ejec_exp_mgcg_prelim_dip
"""
work_conn.execute(text(sql_ejec_exp_mgcg_dip))
ejec_exp_mgcg_dip = pd.read_sql(text("SELECT * FROM #ejec_exp_mgcg_dip"), work_conn)
_log("ejec_exp_mgcg_dip", ejec_exp_mgcg_dip)


In [ ]:
# Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgcg_total_dip"))
sql_ejec_mgcg_total_dip = """
SELECT * INTO #ejec_mgcg_total_dip FROM #ejec_muestra_mgcg_dip
UNION ALL
SELECT * FROM #ejec_exp_mgcg_dip
"""
work_conn.execute(text(sql_ejec_mgcg_total_dip))
ejec_mgcg_total_dip = pd.read_sql(text("SELECT * FROM #ejec_mgcg_total_dip"), work_conn)
_log("ejec_mgcg_total_dip", ejec_mgcg_total_dip)


In [ ]:
# DIPRES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF trim
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mgcg_aper_dip"))
sql_ejec_y_mgcg_aper_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_y_mgcg_aper_dip
FROM #ejec_mgcg_total_dip t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA,t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,t1.C_ENTIDAD,t1.C_PARTIDA,t1.C_CAPITULO,t1.C_PROGRAMA,t1.C_TIPO_CUENTA,t1.C_CUENTA,
         t1.C_ITEM, t1.C_ASIGNACION,t1.C_ANALITICO,t1.N_ANALITICO,t1.C_SCN,t1.C_CAGENTE,t1.ACTIVIDAD,t1.NIVEL
"""
work_conn.execute(text(sql_ejec_y_mgcg_aper_dip))
ejec_y_mgcg_aper_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mgcg_aper_dip"), work_conn)
_log("ejec_y_mgcg_aper_dip", ejec_y_mgcg_aper_dip)


In [ ]:
# DIPRES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para CNF trim
sql_ejec_y_mgcg_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA,t1.AÑO,t1.TRIM,t1.MES,
       t1.C_TIPO_CUENTA,  t1.C_CUENTA,t1.C_ITEM,
       t1.C_SCN,t1.C_CAGENTE,
       SUM(t1.dato) AS Dato,t1.ACTIVIDAD,t1.NIVEL
FROM #ejec_y_mgcg_aper_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA,t1.AÑO,t1.TRIM,t1.MES,
         t1.C_TIPO_CUENTA,  t1.C_CUENTA,t1.C_ITEM,
         t1.C_SCN,t1.C_CAGENTE,t1.ACTIVIDAD,t1.NIVEL
"""
ejec_y_mgcg_dip = pd.read_sql(text(sql_ejec_y_mgcg_dip), work_conn)
_log("ejec_y_mgcg_dip", ejec_y_mgcg_dip)


In [ ]:
# PROC SQL; DROP TABLE EST_MGCG_prev_DIP,EJEC_MUEST_MGCG_prev_DIP,EJEC_MUEST_MGCG_DIP,
# ANALIT_TCONS_prev_DIP,CRUCE_EJEC_EXP_MGCG_DIP,EJEC_EXP_MGCG_prelim_DIP,
# EJEC_EXP_MGCG_DIP,EJEC_MGCG_TOTAL_DIP
# Limpieza de temporales de sesión (algunos nombres del SAS original no se crearon en este nodo; se dropean solo los propios)
for _t in ["#ejec_exp_mgcg_prelim_dip", "#ejec_exp_mgcg_dip", "#ejec_mgcg_total_dip", "#cruce_ejec_exp_mgcg_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC TC MM_DIP

Redistribuye la muestra y expande el resto de transacciones MM según estructura de ponderación (POND) y promedios mensuales, generando la ejecución total y sus agregados trimestrales por SCN/agente para el análisis de flujo financiero

*confianza: medium · verificador: approve · SAS: PROC SQL LEFT JOIN + agregación con SUM, DATA step de filtrado condicional, UNION ALL y CASE WHEN para trimestre*

In [ ]:
# ========= OPC TC MM_DIP =========
# DIPRES: CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MM SEGUN ESTRUCTURA, PARA EJECUCION AÑO, MES y PACAPR faltante
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mm_dip"))
sql_ejec_muest_mm_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG/1000000*t1.POND) AS DEVENGO
INTO #ejec_muest_mm_dip
FROM GOBGENER.dbo.EST_MM t1
LEFT JOIN #ejec_dip t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
     (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('01','03','08','09') AND t1.ENTIDAD NOT IN (5601,10201)
GROUP BY t2.SUBSECTOR,t2.MONEDA,t2.MES,t2.AÑO,t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.C_ITEM,
         t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t2.C_ENTIDAD,t2.ACTIVIDAD,t2.NIVEL,t1.C_SCN,t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muest_mm_dip))


In [ ]:
# DATA step: if devengo<>0 (DROP=DEVENG) -- el faltante SAS es NULL en SQL Server
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mm_dip2"))
sql_filtro_muest = """
SELECT *
INTO #ejec_muest_mm_dip2
FROM #ejec_muest_mm_dip
WHERE DEVENGO IS NOT NULL AND DEVENGO <> 0
"""
work_conn.execute(text(sql_filtro_muest))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mm_dip"))
work_conn.execute(text("EXEC sp_rename '#ejec_muest_mm_dip2', '#ejec_muest_mm_dip'"))
ejec_muest_mm_dip = pd.read_sql(text("SELECT * FROM #ejec_muest_mm_dip"), work_conn)
_log("ejec_muest_mm_dip", ejec_muest_mm_dip)


In [ ]:
# DIPRES: CREA TABLA DE ANALITICOS DE MM A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mm_dip"))
sql_cruce_ejec_exp_mm_dip = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mm_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip t2 ON (t2.C_ENTIDAD=t1.ENTIDAD AND t2.C_ASIGNACION=t1.C_ASIGNACION AND t2.C_ANALITICO=t1.C_ANALITICO AND t2.C_CUENTA=t1.C_CUENTA AND
            t2.C_ITEM=t1.C_ITEM)
WHERE t1.M_MUESTRA=0 AND t1.Clasif_Tcons='MM' AND (t2.C_ENTIDAD IS NOT NULL AND t2.C_ENTIDAD NOT IN (5601,10201))
"""
work_conn.execute(text(sql_cruce_ejec_exp_mm_dip))
cruce_ejec_exp_mm_dip = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mm_dip"), work_conn)
_log("cruce_ejec_exp_mm_dip", cruce_ejec_exp_mm_dip)


In [ ]:
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES (MM), PARA EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm_dip"))
sql_ejec_exp_mm_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mm_dip
FROM GOBGENER.dbo.EST_EXP_MM t1
LEFT JOIN #cruce_ejec_exp_mm_dip t2 ON (t2.C_CUENTA=t1.C_CUENTA AND t2.C_ITEM=t1.C_ITEM AND t2.C_ASIGNACION=t1.C_ASIGNACION
     AND t2.C_ANALITICO=t1.C_ANALITICO AND t2.C_ENTIDAD=t1.ENTIDAD AND t2.MES=t1.MES)
LEFT JOIN #prom_mes t3 ON (t2.MONEDA = t3.MONEDA) AND (t2.MES = t3.MES) AND (t2.AÑO = t3.AÑO)
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR,t2.MONEDA, t2.AÑO, t2.MES,t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.C_ITEM,
         t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t2.C_ENTIDAD,t2.ACTIVIDAD,t2.NIVEL,t2.ENTIDAD,t2.Clasif_Tcons,t1.C_SCN,
         t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_exp_mm_dip))


In [ ]:
# DATA step: if devengo<>0 (DROP=DEVENG DATO_OR Clasif_Tcons ENTIDAD)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm_dip2"))
sql_filtro_exp = """
SELECT *
INTO #ejec_exp_mm_dip2
FROM #ejec_exp_mm_dip
WHERE DEVENGO IS NOT NULL AND DEVENGO <> 0
"""
work_conn.execute(text(sql_filtro_exp))
work_conn.execute(text("ALTER TABLE #ejec_exp_mm_dip2 DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons, ENTIDAD"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm_dip"))
work_conn.execute(text("EXEC sp_rename '#ejec_exp_mm_dip2', '#ejec_exp_mm_dip'"))
ejec_exp_mm_dip = pd.read_sql(text("SELECT * FROM #ejec_exp_mm_dip"), work_conn)
_log("ejec_exp_mm_dip", ejec_exp_mm_dip)


In [ ]:
# Crea tabla con la ejecucion total de transacciones MM, Muestra + Resto
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mm_total_dip"))
sql_ejec_mm_total_dip = """
SELECT * INTO #ejec_mm_total_dip FROM #ejec_muest_mm_dip
UNION ALL
SELECT * FROM #ejec_exp_mm_dip
"""
work_conn.execute(text(sql_ejec_mm_total_dip))
ejec_mm_total_dip = pd.read_sql(text("SELECT * FROM #ejec_mm_total_dip"), work_conn)
_log("ejec_mm_total_dip", ejec_mm_total_dip)


In [ ]:
# DIPRES: SALIDA de EJECUCION TOTAL para CNF trim desagregada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mm_aper_dip"))
sql_ejec_y_mm_aper_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_y_mm_aper_dip
FROM #ejec_mm_total_dip t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA,t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,t1.C_ENTIDAD,t1.C_PARTIDA,t1.C_CAPITULO,t1.C_PROGRAMA,t1.C_TIPO_CUENTA,t1.C_CUENTA,
         t1.C_ITEM, t1.C_ASIGNACION,t1.C_ANALITICO,t1.N_ANALITICO,t1.C_SCN,t1.C_CAGENTE,t1.ACTIVIDAD,t1.NIVEL
"""
work_conn.execute(text(sql_ejec_y_mm_aper_dip))
ejec_y_mm_aper_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mm_aper_dip"), work_conn)
_log("ejec_y_mm_aper_dip", ejec_y_mm_aper_dip)


In [ ]:
# DIPRES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES para CNF trim, agregada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mm_dip"))
sql_ejec_y_mm_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA,t1.AÑO,t1.TRIM,t1.MES,
       t1.C_TIPO_CUENTA, t1.C_CUENTA,t1.C_ITEM,
       t1.C_SCN,t1.C_CAGENTE,
       SUM(t1.Dato) AS Dato,t1.ACTIVIDAD,t1.NIVEL
INTO #ejec_y_mm_dip
FROM #ejec_y_mm_aper_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA,t1.AÑO,t1.TRIM,t1.MES,
         t1.C_TIPO_CUENTA, t1.C_CUENTA,t1.C_ITEM,
         t1.C_SCN,t1.C_CAGENTE,t1.ACTIVIDAD,t1.NIVEL
"""
work_conn.execute(text(sql_ejec_y_mm_dip))
ejec_y_mm_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mm_dip"), work_conn)
_log("ejec_y_mm_dip", ejec_y_mm_dip)


In [ ]:
# DROP TABLE de limpieza final: tablas de sesion que ya cumplieron su rol
for _t in ["#ejec_muest_mm_dip", "#cruce_ejec_exp_mm_dip", "#ejec_exp_mm_dip", "#ejec_mm_total_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC TC MGG_DIP

Redistribuye la muestra de transacciones MGG según la estructura de ponderación por entidad/cuenta/item, expande el resto de transacciones fuera de muestra y consolida la ejecución trimestral MGG por SCN, excluyendo entidades y códigos SCN fuera de alcance

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOINs contra estructura de ponderación + agregación con CASE inline (CALCULATED TRIM) + UNION ALL muestra/resto + DROP TABLE final*

In [ ]:
# ========= OPC TC MGG_DIP =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mgg_dip"))
# DIPRES: CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGG SEGUN ESTRUCTURA, PARA EJECUCION AÑO, MES y PACAPR faltante
sql_ejec_muest_mgg_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG/1000000*t1.POND) AS DEVENGO
INTO #ejec_muest_mgg_dip
FROM GOBGENER.dbo.EST_MGG t1
LEFT JOIN #ejec_dip t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
        (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
WHERE t2.C_CUENTA IN ('24') AND t2.C_ITEM IN ('01','03','08','09') AND t1.ENTIDAD NOT IN (5601,10201)
GROUP BY t2.SUBSECTOR,t2.MONEDA,t2.MES,t2.AÑO,t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.C_ITEM,
         t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t2.C_ENTIDAD,t2.ACTIVIDAD,t2.NIVEL,t1.C_SCN,t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muest_mgg_dip))


In [ ]:
# DATA EJEC_MUEST_MGG_DIP (DROP=DEVENG); if devengo<>0 -> filtra y descarta la columna de apoyo, server-side
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mgg_dip_2"))
work_conn.execute(text("""
SELECT * INTO #ejec_muest_mgg_dip_2 FROM #ejec_muest_mgg_dip WHERE DEVENGO <> 0
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mgg_dip"))
cols_ejec_muest_mgg_dip = [c for c in pd.read_sql(text("SELECT TOP 0 * FROM #ejec_muest_mgg_dip_2"), work_conn).columns if c != "DEVENG"]
cols_sel_ejec_muest = ", ".join(cols_ejec_muest_mgg_dip)
work_conn.execute(text(f"SELECT {cols_sel_ejec_muest} INTO #ejec_muest_mgg_dip FROM #ejec_muest_mgg_dip_2"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mgg_dip_2"))
ejec_muest_mgg_dip = pd.read_sql(text("SELECT * FROM #ejec_muest_mgg_dip"), work_conn)
_log("ejec_muest_mgg_dip", ejec_muest_mgg_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgg_dip"))
# DIPRES: CREA TABLA DE ANALITICOS DE MGG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
# El faltante de SAS (t2.C_ENTIDAD=.) tras el LEFT JOIN es NULL en SQL Server
sql_cruce_ejec_exp_mgg_dip = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgg_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip t2 ON (t2.C_ENTIDAD=t1.ENTIDAD AND t2.C_ASIGNACION=t1.C_ASIGNACION AND t2.C_ANALITICO=t1.C_ANALITICO AND t2.C_CUENTA=t1.C_CUENTA AND
                            t2.C_ITEM=t1.C_ITEM)
WHERE t1.M_MUESTRA=0 AND t1.Clasif_Tcons='MGG' AND (t2.C_ENTIDAD IS NOT NULL AND t2.C_ENTIDAD NOT IN (5601,10201))
"""
work_conn.execute(text(sql_cruce_ejec_exp_mgg_dip))
cruce_ejec_exp_mgg_dip = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mgg_dip"), work_conn)
_log("cruce_ejec_exp_mgg_dip", cruce_ejec_exp_mgg_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_dip"))
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES, PARA EJECUCION AÑO t
sql_ejec_exp_mgg_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mgg_dip
FROM GOBGENER.dbo.EST_EXP_MGG t1
LEFT JOIN #cruce_ejec_exp_mgg_dip t2 ON (t2.C_CUENTA=t1.C_CUENTA AND t2.C_ITEM=t1.C_ITEM AND t2.MES=t1.MES)
LEFT JOIN #prom_mes t3 ON (t2.MONEDA = t3.MONEDA) AND (t2.MES = t3.MES) AND (t2.AÑO = t3.AÑO)
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR,t2.MONEDA, t2.AÑO, t2.MES,t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.C_ITEM,
         t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t2.C_ENTIDAD,t2.ACTIVIDAD,t2.NIVEL,t2.ENTIDAD,t2.Clasif_Tcons,t1.C_SCN,
         t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_exp_mgg_dip))


In [ ]:
# DATA EJEC_EXP_MGG_DIP (DROP=DEVENG DATO_OR Clasif_Tcons ENTIDAD); if devengo<>0
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_dip_2"))
work_conn.execute(text("SELECT * INTO #ejec_exp_mgg_dip_2 FROM #ejec_exp_mgg_dip WHERE DEVENGO <> 0"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_dip"))
_cols_drop_exp_mgg = {"DEVENG", "DATO_OR", "Clasif_Tcons", "ENTIDAD"}
cols_ejec_exp_mgg_dip = [c for c in pd.read_sql(text("SELECT TOP 0 * FROM #ejec_exp_mgg_dip_2"), work_conn).columns if c not in _cols_drop_exp_mgg]
cols_sel_exp_mgg = ", ".join(cols_ejec_exp_mgg_dip)
work_conn.execute(text(f"SELECT {cols_sel_exp_mgg} INTO #ejec_exp_mgg_dip FROM #ejec_exp_mgg_dip_2"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_dip_2"))
ejec_exp_mgg_dip = pd.read_sql(text("SELECT * FROM #ejec_exp_mgg_dip"), work_conn)
_log("ejec_exp_mgg_dip", ejec_exp_mgg_dip)


In [ ]:
# Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgg_total_dip"))
work_conn.execute(text("""
SELECT * INTO #ejec_mgg_total_dip FROM #ejec_muest_mgg_dip
UNION ALL
SELECT * FROM #ejec_exp_mgg_dip
"""))
ejec_mgg_total_dip = pd.read_sql(text("SELECT * FROM #ejec_mgg_total_dip"), work_conn)
_log("ejec_mgg_total_dip", ejec_mgg_total_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mgg_aper_dip"))
# DIPRES: SALIDA de EJECUCION TOTAL para CNF trim desagregada
# TRIM es función pura del MES (agrupado): se calcula sobre el resultado agregado, sin CALCULATED
sql_ejec_y_mgg_aper_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_y_mgg_aper_dip
FROM #ejec_mgg_total_dip t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA,t1.AÑO,
         CASE WHEN t1.MES IN (1,2,3) THEN 1
              WHEN t1.MES IN (4,5,6) THEN 2
              WHEN t1.MES IN (7,8,9) THEN 3
              ELSE 4 END,
         t1.MES,t1.C_ENTIDAD,t1.C_PARTIDA,t1.C_CAPITULO,t1.C_PROGRAMA,t1.C_TIPO_CUENTA,t1.C_CUENTA,
         t1.C_ITEM, t1.C_ASIGNACION,t1.C_ANALITICO,t1.N_ANALITICO,t1.C_SCN,t1.C_CAGENTE,t1.ACTIVIDAD,t1.NIVEL
"""
work_conn.execute(text(sql_ejec_y_mgg_aper_dip))
ejec_y_mgg_aper_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mgg_aper_dip"), work_conn)
_log("ejec_y_mgg_aper_dip", ejec_y_mgg_aper_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_y_mgg_dip"))
# DIPRES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES para CNF trim, agregada
sql_ejec_y_mgg_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA,t1.AÑO,t1.TRIM,t1.MES,
       t1.C_TIPO_CUENTA, t1.C_CUENTA,t1.C_ITEM,
       t1.C_SCN,t1.C_CAGENTE,
       SUM(t1.Dato) AS Dato,t1.ACTIVIDAD,t1.NIVEL
INTO #ejec_y_mgg_dip
FROM #ejec_y_mgg_aper_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA,t1.AÑO,t1.TRIM,t1.MES,
         t1.C_TIPO_CUENTA, t1.C_CUENTA,t1.C_ITEM,
         t1.C_SCN,t1.C_CAGENTE,t1.ACTIVIDAD,t1.NIVEL
"""
work_conn.execute(text(sql_ejec_y_mgg_dip))
ejec_y_mgg_dip = pd.read_sql(text("SELECT * FROM #ejec_y_mgg_dip"), work_conn)
_log("ejec_y_mgg_dip", ejec_y_mgg_dip)


In [ ]:
# DROP TABLE de WORKs intermedios del SAS: nombres no materializados en esta traducción
# (EST_MGG_prev_DIP, EJEC_MUEST_MGG_prev_DIP, ANALIT_TCONS_prev_DIP, EJEC_EXP_MGG_prelim_DIP no existen como #tmp acá)
for _t in ["#ejec_muest_mgg_dip", "#cruce_ejec_exp_mgg_dip", "#ejec_exp_mgg_dip", "#ejec_mgg_total_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC APER_CNF_trim (solo DIP)

Ejecucion CNF trimestral desagregada y agregada de DIPRES: clasifica analiticos fijos y variables por transaccion SCN, aplica reasignaciones de negocio (Transantiago, cobre bruto, becas, ayudas IPS, nivel de gobierno), consolida en una sola tabla y elimina duplicidades/transacciones redundantes de la consolidacion

*confianza: medium · verificador: approve · SAS: PROC SQL: analiticos predefinidos/variables con CASE WHEN, LEFT JOINs a CR18_T_SCN_1/2, UPDATE/DELETE de reasignaciones negociales, UNION de 6 tablas (A-F, Y) y agregacion final*

In [ ]:
# ========= OPC APER_CNF_trim (solo DIP) =========
# DIPRES: Salida para ANALISIS DESAGREGADO de indicadores GC de CNF trim DATA DE DIPRES (mes t)
# DIPRES: Analiticos predefinidos
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_ejecucion_a_prelim_dip_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG*t3.DATO/1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper))


In [ ]:
# nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA='07' AND C_ITEM IN ('0','00')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (ingresos operacionales agregados)", res.rowcount)

# Analiticos predefinidos: AJUSTE COYUNTURA - se reasigna de indicador transaccion CORFO
# que se desagrega anualmente entre ventas y D45
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper "
    "WHERE C_ENTIDAD=3801 AND C_CUENTA='06' AND C_ITEM='01' AND C_ASIGNACION='000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper (ajuste coyuntura CORFO)", res.rowcount)

res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper (cuenta 32 analiticos)", res.rowcount)


In [ ]:
# Analiticos predefinidos: reasignacion D39 2% constitucional 2008 para Transantiago
_where_transantiago = (
    "C_ENTIDAD=19703 AND C_CUENTA='26' AND C_ITEM='03' AND C_ASIGNACION='000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA IN ('P') AND AÑO=2008 AND MES IN (10,11,12)"
)
res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_dip_aper SET C_CAGENTE='S11' WHERE {_where_transantiago}"))
_log("UPDATE #ejecucion_a_prelim_dip_aper C_CAGENTE (Transantiago)", res.rowcount)

res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_dip_aper SET C_SCN='D39' WHERE {_where_transantiago}"))
_log("UPDATE #ejecucion_a_prelim_dip_aper C_SCN (Transantiago)", res.rowcount)

res = work_conn.execute(text(f"UPDATE #ejecucion_a_prelim_dip_aper SET C_TRANSACCION='4139' WHERE {_where_transantiago}"))
_log("UPDATE #ejecucion_a_prelim_dip_aper C_TRANSACCION (Transantiago)", res.rowcount)


In [ ]:
# Analiticos predefinidos: AJUSTE COBRE BRUTO (Financiamiento Capacidad Estrategicas Defensa Nacional)
# reasignacion entre D75 y D42
res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_dip_aper SET C_SCN='D42', C_TRANSACCION='42222', N_CUENTA='Cobre bruto' "
    "WHERE C_ENTIDAD=19701 AND C_CUENTA='08' AND C_ITEM='99' AND C_ASIGNACION='999' "
    "AND C_ANALITICO IN ('99999000') AND MONEDA IN ('D') AND AÑO>2019"
))
_log("UPDATE #ejecucion_a_prelim_dip_aper (ajuste cobre bruto)", res.rowcount)

# Se eliminan Intereses en moneda dolar, no se consideran en indicador CR18
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_SCN='D41' AND MONEDA='D'"))
_log("DELETE #ejecucion_a_prelim_dip_aper (intereses moneda dolar)", res.rowcount)

# Se eliminan dividendos tesoro, debido a que se toma dato de sectorialistas
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_SCN IN ('D42','D421') AND C_PARTIDA='50' AND C_CUENTA NOT IN ('08')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper (dividendos tesoro)", res.rowcount)


In [ ]:
# DIPRES: Analiticos predefinidos (sigue)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_ejecucion_a_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL,
       SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper))


In [ ]:
# DIPRES: Analiticos FIJOS CUENTA 32
# t2.C_SCN viene de un LEFT JOIN: en SAS, NOT IN contra un valor missing evalua a falso
# y la fila queda fuera del WHERE -> se replica con INNER JOIN implicito via filtro
# t2.C_SCN IS NOT NULL AND NOT IN (...), sin agregar de vuelta las filas sin match
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_ejecucion_b_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG*t3.DATO/1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA='32' AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN IS NOT NULL
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper))


In [ ]:
# analiticos variables
# Mismo criterio: t2.OBS y t2.C_SCN vienen del LEFT JOIN; SAS descarta la fila cuando
# t2 no matchea (NOT IN de missing = unknown => WHERE falso). Se exige match (IS NOT NULL)
# antes de aplicar los NOT IN, sin reincorporar las filas sin match.
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_prelim_aper"))
sql_ejecucion_c_dip_prelim_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG*t3.DATO/1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_prelim_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.OBS IS NOT NULL AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact')
  AND t2.C_SCN IS NOT NULL AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9','')
"""
work_conn.execute(text(sql_ejecucion_c_dip_prelim_aper))


In [ ]:
# Analiticos variables: reasignacion contragente ayudas IPS
# Bono Ayuda Familiar (18O)
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_dip_prelim_aper SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=24101 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') AND AÑO=2019 AND MES=12"
))
_log("UPDATE #ejecucion_c_dip_prelim_aper (Bono Ayuda Familiar)", res.rowcount)

# Bono Covid
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_dip_prelim_aper SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=24101 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') AND AÑO=2020 AND MES=4"
))
_log("UPDATE #ejecucion_c_dip_prelim_aper (Bono Covid)", res.rowcount)

# Reasignacion nivel de gobierno para Programa Ampliado de Inmunizaciones
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_dip_prelim_aper SET NIVEL='MUNICIPAL' "
    "WHERE C_ENTIDAD IN (11105,22701,22750) AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='004'"
))
_log("UPDATE #ejecucion_c_dip_prelim_aper (PAI)", res.rowcount)

# Reasignacion nivel de D63 municipal
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_dip_prelim_aper SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='REGIONAL' AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM='03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))
_log("UPDATE #ejecucion_c_dip_prelim_aper (D63 municipal)", res.rowcount)

# Beca Especial Carbon (previo 2020)
_where_beca_carbon = (
    "C_ENTIDAD=6403 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO<2020"
)
res = work_conn.execute(text(f"UPDATE #ejecucion_c_dip_prelim_aper SET C_CAGENTE='S14' WHERE {_where_beca_carbon}"))
_log("UPDATE #ejecucion_c_dip_prelim_aper C_CAGENTE (Beca Carbon)", res.rowcount)

res = work_conn.execute(text(f"UPDATE #ejecucion_c_dip_prelim_aper SET C_SCN='D75' WHERE {_where_beca_carbon}"))
_log("UPDATE #ejecucion_c_dip_prelim_aper C_SCN (Beca Carbon)", res.rowcount)

res = work_conn.execute(text(f"UPDATE #ejecucion_c_dip_prelim_aper SET C_TRANSACCION='4345' WHERE {_where_beca_carbon}"))
_log("UPDATE #ejecucion_c_dip_prelim_aper C_TRANSACCION (Beca Carbon)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_ejecucion_c_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, t1.Dato
INTO #ejecucion_c_dip_aper
FROM #ejecucion_c_dip_prelim_aper t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper))


In [ ]:
# DIPRES: Transacciones diferenciadas para coyuntura
# t2.OBS y t2.C_ACTIVO vienen de un LEFT JOIN a CR18_T_SCN_2; el WHERE exige t2.OBS='CR18_difcoy'
# (igualdad directa, no NOT IN): con t2 sin match OBS es NULL y la igualdad ya descarta la fila
# sin necesidad de tratamiento especial (comportamiento identico en SAS y SQL Server)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_ejecucion_d_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG*t3.DATO/1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS = 'CR18_difcoy' AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN IS NOT NULL AND t2.C_SCN NOT IN ('TC')
  AND t2.C_ACTIVO IS NOT NULL
  AND t2.C_ACTIVO NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper))

# Reasignacion nivel de D63 municipal
res = work_conn.execute(text(
    "UPDATE #ejecucion_d_dip_aper SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='REGIONAL' AND C_ACTIVO IN ('D63') AND C_ITEM='03'"
))
_log("UPDATE #ejecucion_d_dip_aper (D63 municipal)", res.rowcount)


In [ ]:
# DIPRES: Transacciones diferenciadas por actividad
# Igual que D: t2.OBS='CR18_difact' es igualdad directa (no NOT IN), y con t2 sin match
# OBS es NULL -> la igualdad descarta la fila igual que en SAS. t2.C_SCN sí usa NOT IN
# y requiere match (IS NOT NULL) para replicar el descarte de missing de SAS.
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_ejecucion_f_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT, t1.NIVEL,
       CAST((t1.DEVENG*t3.DATO/1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS = 'CR18_difact' AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN IS NOT NULL AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN NOT IN ('D73','D93','F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper))


In [ ]:
# DIPRES: UNE TABLAS de TCONS Dipres
# EJEC_Y_MIXT_aper_DIP, EJEC_Y_MGCG_aper_DIP, EJEC_Y_MM_aper_DIP, EJEC_Y_MGG_aper_DIP son WORK
# dejados por nodos previos (session_sql): se materializan como #tmp para el UNION server-side
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_y_dip_prev_aper"))
sql_ejecucion_y_dip_prev_aper = """
SELECT * INTO #ejecucion_y_dip_prev_aper FROM #ejec_y_mixt_aper_dip
UNION ALL SELECT * FROM #ejec_y_mgcg_aper_dip
UNION ALL SELECT * FROM #ejec_y_mm_aper_dip
UNION ALL SELECT * FROM #ejec_y_mgg_aper_dip
"""
work_conn.execute(text(sql_ejecucion_y_dip_prev_aper))

# DATA EJECUCION_Y_DIP_prev_aper; SET EJECUCION_Y_DIP_prev_aper; C_ACT=ACTIVIDAD; C_ACTIVO='TC';
res = work_conn.execute(text("UPDATE #ejecucion_y_dip_prev_aper SET C_ACT = ACTIVIDAD, C_ACTIVO = 'TC'"))
_log("UPDATE #ejecucion_y_dip_prev_aper (C_ACT/C_ACTIVO)", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_y_dip_prev_aper SET C_ACT='AAPP' "
    "WHERE ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST')"
))
_log("UPDATE #ejecucion_y_dip_prev_aper (C_ACT AAPP)", res.rowcount)


In [ ]:
# DATA EJECUCION_Y_DIP_aper (DROP=ACTIVIDAD); SET EJECUCION_Y_DIP_prev_aper;
#   IF C_SCN NOT IN (...) -- subsetting IF: aca no hay JOIN de por medio (C_SCN ya viene
#   resuelto en la propia tabla), por lo tanto NOT IN se comporta igual con o sin missing
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_y_dip_aper"))
cols_ejecucion_y_dip_aper = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, "
    "C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
sql_ejecucion_y_dip_aper = f"""
SELECT {cols_ejecucion_y_dip_aper}
INTO #ejecucion_y_dip_aper
FROM #ejecucion_y_dip_prev_aper
WHERE C_SCN NOT IN ('D73','D93','F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
work_conn.execute(text(sql_ejecucion_y_dip_aper))


In [ ]:
# DIPRES: UNE TABLAS de Salida total de ejecucion CNF trim de data DIPRES, DESAGREGADA
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_dip_aper"))
sql_ejecucion_cnf_dip_aper = """
SELECT * INTO #ejecucion_cnf_dip_aper FROM #ejecucion_a_dip_aper
UNION ALL SELECT * FROM #ejecucion_b_dip_aper
UNION ALL SELECT * FROM #ejecucion_c_dip_aper
UNION ALL SELECT * FROM #ejecucion_d_dip_aper
UNION ALL SELECT * FROM #ejecucion_y_dip_aper
UNION ALL SELECT * FROM #ejecucion_f_dip_aper
"""
work_conn.execute(text(sql_ejecucion_cnf_dip_aper))

res = work_conn.execute(text("UPDATE #ejecucion_cnf_dip_aper SET N_CUENTA='DIPRES'"))
_log("UPDATE #ejecucion_cnf_dip_aper (N_CUENTA DIPRES)", res.rowcount)

# DROP de las tablas intermedias que el SAS ya no necesita (equivalente: limpiar las #tmp)
for _t in ["#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper",
           "#ejecucion_c_dip_aper", "#ejecucion_c_dip_prelim_aper", "#ejecucion_d_dip_aper",
           "#ejecucion_f_dip_aper", "#ejecucion_y_dip_aper", "#ejecucion_y_dip_prev_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# Reasignacion de P3 para cuentas de subsector
res = work_conn.execute(text("UPDATE #ejecucion_cnf_dip_aper SET C_SCN='D73' WHERE C_SCN='P3'"))
_log("UPDATE #ejecucion_cnf_dip_aper (P3->D73)", res.rowcount)

# Reasignacion de nivel regional a central, por metodologia de acuerdo a caracteristicas actuales de GOREs
res = work_conn.execute(text("UPDATE #ejecucion_cnf_dip_aper SET NIVEL='CENTRAL' WHERE NIVEL='REGIONAL'"))
_log("UPDATE #ejecucion_cnf_dip_aper (regional->central)", res.rowcount)

# Reasignacion FBCF realizada por municipalidades pero medida en ejecucion GC
res = work_conn.execute(text(
    "UPDATE #ejecucion_cnf_dip_aper SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='CENTRAL' AND C_SCN='P51' AND C_CAGENTE='S1313'"
))
_log("UPDATE #ejecucion_cnf_dip_aper (FBCF municipal)", res.rowcount)

# Eliminacion de transacciones redundantes en consolidacion
for _scn, _nivel, _cagente in [
    ("D73", "CENTRAL", "S13"), ("D73", "CENTRAL", "S131"),
    ("D93", "CENTRAL", "S13"), ("D93", "CENTRAL", "S131"),
    ("D73", "CENTRAL", "S1312"), ("D93", "CENTRAL", "S1312"),
    ("D73", "PREVISION", "S1314"), ("D93", "PREVISION", "S1314"),
]:
    res = work_conn.execute(text(
        f"DELETE FROM #ejecucion_cnf_dip_aper WHERE C_SCN='{_scn}' AND NIVEL='{_nivel}' AND C_CAGENTE='{_cagente}'"
    ))
    _log(f"DELETE #ejecucion_cnf_dip_aper ({_scn}/{_nivel}/{_cagente})", res.rowcount)


In [ ]:
# NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES,
# ASI SE MEJORA EL INDICADOR DE VENTAS
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
sql_ejecucion_a_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG*t3.DATO/1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_dip_07))
ejecucion_a_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_07"), work_conn)
_log("ejecucion_a_dip_07", ejecucion_a_dip_07)


In [ ]:
# Verifica que entidades informan solo el agregado para filtrarlas
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA
"""
work_conn.execute(text(sql_cuenta_item_07))
cuenta_item_07 = pd.read_sql(text("SELECT * FROM #cuenta_item_07"), work_conn)
_log("cuenta_item_07", cuenta_item_07)


In [ ]:
# t2.ID=1 es un INNER JOIN implicito: solo entidades que informan un unico registro
# agregado por cuenta/item (sin ambiguedad de NULLs, ya que ID es COUNT y nunca falta match)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
sql_ejecucion_dip_07 = """
SELECT DISTINCT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD,
       t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN,
       t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, 'DIPRES' AS N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
  ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
     AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
WHERE t2.ID = 1
"""
work_conn.execute(text(sql_ejecucion_dip_07))
ejecucion_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_dip_07"), work_conn)
_log("ejecucion_dip_07", ejecucion_dip_07)


In [ ]:
# une imputacion de ingresos operacionales
sql_une_ejecucion_cnf_dip_aper = """
INSERT INTO #ejecucion_cnf_dip_aper
SELECT * FROM #ejecucion_dip_07
"""
res = work_conn.execute(text(sql_une_ejecucion_cnf_dip_aper))
_log("INSERT #ejecucion_cnf_dip_aper (imputacion ingresos operacionales)", res.rowcount)

for _t in ["#ejecucion_a_dip_07", "#cuenta_item_07", "#ejecucion_dip_07"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# SALIDA AGREGADA DIPRES
# EJECUCION_CNF_DIP no viene en created_tables ni es tabla de BD del catalogo:
# es un dataset WORK final del flujo, se materializa server-side y se baja a pandas
# para que quede disponible al notebook (output_datasets)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_dip"))
sql_ejecucion_cnf_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.Dato) AS DATO
INTO #ejecucion_cnf_dip
FROM #ejecucion_cnf_dip_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACT, t1.C_CUP, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_cnf_dip))
ejecucion_cnf_dip = pd.read_sql(text("SELECT * FROM #ejecucion_cnf_dip"), work_conn)
_log("ejecucion_cnf_dip", ejecucion_cnf_dip)
